# Sprzętowy Generator Kształtów 3D - Stożek

Ten notatnik służy do weryfikacji sprzętowego akceleratora 3D wdrożonego na układzie FPGA platformy Kria KV260. System wykorzystuje procesor ARM (Zynq MPSoC) do orkiestracji oraz dedykowany moduł IP `cordic_3d_0` zaimplementowany w logice programowalnej, który sprzętowo generuje wierzchołki przestrzenne stożka.

Dane geometryczne są wysyłane strumieniowo ze sprzętu bezpośrednio do pamięci RAM procesora z wykorzystaniem interfejsu AXI Direct Memory Access (DMA), co zapewnia maksymalną przepustowość.

**Przepływ danych:**
1. Konfiguracja akceleratora (AXI4-Lite).
2. Alokacja bufora fizycznego w RAM.
3. Sprzętowa generacja i transfer współrzędnych (AXI4-Stream -> DMA).
4. Przetworzenie sprzętowych typów stałoprzecinkowych na liczby zmiennoprzecinkowe.
5. Wizualizacja 3D kształtu w środowisku Jupyter.

In [ ]:
# Import niezbędnych bibliotek
from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Magiczna komenda Jupyter do wyświetlania wykresów bezpośrednio w notatniku
%matplotlib inline

print("Biblioteki zostały pomyślnie zaimportowane.")

In [ ]:
# Załadowanie pliku bitstream (overlay) do matrycy FPGA
print("Wgrywanie strumienia bitowego do FPGA...")
overlay = Overlay("cordic_3d.xsa")

# Przypisanie wskaźników (aliasów) do sprzętowych bloków IP
dma = overlay.axi_dma_0
cordic_ctrl = overlay.cordic_3d_0

print("Zakończono. Moduły sprzętowe (DMA i CORDIC 3D) są gotowe do pracy.")

In [ ]:
# Konfiguracja akceleratora przez interfejs AXI4-Lite
# UWAGA: Poniższe adresy offsetów zależą ściśle od Twojej mapy rejestrów wygenerowanej w Vivado.
# Zazwyczaj pierwszy rejestr to 0x00, drugi 0x04 itd.

MODE_REG_OFFSET = 0x00    # np. rejestr wyboru kształtu
PARAM_REG_OFFSET = 0x04   # np. rejestr definiujący wysokość lub gęstość siatki

print("Konfigurowanie akceleratora CORDIC_3D...")

# Zapis do rejestrów (Wirtualne ustawienie: Tryb = 1 [Stożek], Parametr = 3000)
cordic_ctrl.write(MODE_REG_OFFSET, 1)
cordic_ctrl.write(PARAM_REG_OFFSET, 3000)

print(f"Konfiguracja zapisana. Odczyt kontrolny z MODE_REG: {cordic_ctrl.read(MODE_REG_OFFSET)}")

In [ ]:
# Alokacja ciągłego bloku pamięci RAM dostępnego dla DMA
vertices_count = 3000
values_per_vertex = 3 # Trzy wymiary: X, Y, Z
buffer_size = vertices_count * values_per_vertex

print(f"Alokacja bufora dla {vertices_count} wierzchołków ({buffer_size} wartości int32)...")

# pynq.allocate gwarantuje ciągłość fizyczną w RAM, wymaganą przez DMA
out_buffer = allocate(shape=(buffer_size,), dtype=np.int32)

print("Pamięć zaalokowana poprawnie.")

In [ ]:
print("Rozpoczynanie sprzętowej generacji i transferu AXI-Stream -> DMA...")

# Uruchomienie nasłuchiwania kanału odbiorczego (S2MM)
dma.recvchannel.transfer(out_buffer)

# Zablokowanie wątku w oczekiwaniu na sygnał TLAST kończący pakiet od akceleratora
dma.recvchannel.wait()

print("Transfer sprzętowy zakończony sukcesem! Dane znajdują się w pamięci RAM.")

In [ ]:
print("Przetwarzanie danych stałoprzecinkowych na format float...")

# Kopiujemy dane do natywnej tablicy numpy i od razu rzutujemy/skalujemy
# Zapobiega to utracie danych podczas zwalniania bufora CMA
data_flat = np.array(out_buffer, copy=True) / 1024.0

# Rozdzielenie płaskiego bufora [X1,Y1,Z1, X2,Y2,Z2...] przy użyciu numpy slicing
X = data_flat[0::3]
Y = data_flat[1::3]
Z = data_flat[2::3]

print(f"Liczba zdekodowanych współrzędnych: X={len(X)}, Y={len(Y)}, Z={len(Z)}\n")
print("Pierwsze 5 zdekodowanych wierzchołków:")
for i in range(min(5, vertices_count)):
    print(f"Wierzchołek {i}: X = {X[i]:.4f}, Y = {Y[i]:.4f}, Z = {Z[i]:.4f}")

In [ ]:
print("Generowanie wykresu 3D...")

# Inicjalizacja wykresu przestrzennego
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Tworzenie wykresu punktowego (Scatter Plot), kolorowanie wzdłuż osi Z dla lepszej głębi
scatter = ax.scatter(X, Y, Z, c=Z, cmap='viridis', s=5, alpha=0.8)

# Ustawienie etykiet
ax.set_title("Sprzętowo wygenerowany Stożek 3D (CORDIC)")
ax.set_xlabel('Oś X')
ax.set_ylabel('Oś Y')
ax.set_zlabel('Oś Z')

# Dodanie paska kolorów
fig.colorbar(scatter, ax=ax, label='Wysokość (Z)', pad=0.1)

# Ustalenie proporcji wykresu, by kształt nie był zniekształcony/rozciągnięty 
# Wymaga wyliczenia rozstępu (peak-to-peak) dla każdej osi
max_range = np.array([X.max()-X.min(), Y.max()-Y.min(), Z.max()-Z.min()]).max()
ax.set_box_aspect((np.ptp(X)/max_range, np.ptp(Y)/max_range, np.ptp(Z)/max_range))

plt.show()

In [ ]:
# Poprawne zarządzanie zasobami: zwolnienie ciągłej pamięci CMA (Contiguous Memory Allocator)
# Brak tego kroku w długodziałających programach doprowadzi do wycieku pamięci fizycznej.
del out_buffer

print("Bufor DMA został pomyślnie zwolniony. Zasoby systemowe przywrócone.")